# Student Memory Store -> Profile Demo

这份 notebook 专门演示一件事：

1. 取三道现成例题
2. 为同一个学生构造 diagnosis / coach / review 事件
3. 把这些事件写入独立的 demo `jsonl` 事件库
4. 再从事件库重建出一个统一的 `student_memory_profile`

这样你可以直接看懂：

- 事件层长什么样
- store 里存了什么
- profile 是怎么从事件里提取出来的

这份 notebook 默认只使用隔离的 demo store，不污染正式事件库。

In [1]:
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from student_memory_events import (
    binding_result_to_memory_event,
    coach_response_to_memory_event,
    diagnosis_result_to_memory_event,
    review_state_update_to_memory_event,
)
from student_memory_store import (
    DEFAULT_EVENTS_DIR,
    append_event,
    build_profile_from_store,
    load_events,
    summarize_store,
)

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'student_memory_store_profile_demo'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
DEMO_EVENTS_PATH = SCRATCH_DIR / 'student_memory_events_demo.jsonl'

def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2))


def reset_demo_store(path: Path):
    if path.exists():
        path.unlink()


def question_binding(primary_node_id, secondary_node_ids):
    return {
        'primary_node_id': primary_node_id,
        'secondary_node_ids': secondary_node_ids,
        'linked_node_ids': [primary_node_id, *secondary_node_ids],
    }


## 1. 准备三道题

这里直接用 `harness/fixtures/wrong_binder` 里的三道现成题：

- 递推构造等比数列
- 平面向量夹角
- 圆心到直线距离范围

为了把重点放在 memory store 上，下面不走真实模型调用，而是手工构造**结构真实、字段完整**的事件。

In [2]:
FIXTURE_DIR = PROJECT_ROOT / 'harness' / 'fixtures' / 'wrong_binder'

CASE_PATHS = [
    FIXTURE_DIR / 'binder_case_sequence_shift.json',
    FIXTURE_DIR / 'binder_case_vector_angle.json',
    FIXTURE_DIR / 'binder_case_circle_distance.json',
]

CASES = [json.loads(path.read_text(encoding='utf-8')) for path in CASE_PATHS]

STUDENT_ID = 'demo_student_memory_store'
SESSION_ID = 'session_memory_store_demo_001'
BASE_TIME = datetime(2026, 6, 24, 20, 0, tzinfo=timezone(timedelta(hours=8)))

CASE_SUMMARY = []
for item in CASES:
    CASE_SUMMARY.append(
        {
            'case_id': item['case_id'],
            'description': item['description'],
            'question_type': item['question_payload']['question_type'],
            'stem': item['question_payload']['stem'],
            'student_answer': item['question_payload']['student_answer'],
        }
    )

show_json(CASE_SUMMARY)


[
  {
    "case_id": "binder_case_sequence_shift",
    "description": "递推构造等比数列证明题",
    "question_type": "证明题",
    "stem": "已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。",
    "student_answer": "我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。"
  },
  {
    "case_id": "binder_case_vector_angle",
    "description": "单位向量和为零，借数量积与夹角公式求夹角",
    "question_type": "选择题",
    "stem": "已知向量 a,b,c 均为单位向量，若 a+b+c=0，则 a 与 b 的夹角为",
    "student_answer": "我能想到数量积，但不会把向量和的模平方展开到夹角公式。"
  },
  {
    "case_id": "binder_case_circle_distance",
    "description": "圆上张角存在性转化为圆心到直线距离范围",
    "question_type": "选择题",
    "stem": "若圆 (x+1)^2+(y-2)^2=1 上存在两个不同的点 M,N，直线 4x+3y+t=0 上存在一点 P，使得 ∠MPN=60°，则实数 t 的取值范围是",
    "student_answer": "我知道要用点到直线距离，但不知道为什么先转成圆心到直线距离不超过 2。"
  }
]


## 2. 为三道题设定绑定与事件脚本

这里我们手工指定：

- 每道题绑定到哪个主知识点 / 次知识点
- diagnosis 判成什么错因
- coach 这轮采用什么策略
- review 的结果是什么

这样做的目的是让事件流清楚、稳定、可复现。

In [3]:
QUESTION_SCRIPTS = [
    {
        'question_id': 'wq_demo_seq_001',
        'case': CASES[0],
        'binding': question_binding(
            'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化',
            ['math.数列与不等式.数列.基础概念.数列递推公式解读'],
        ),
        'diagnosis': {
            'error_type': 'missing_strategy',
            'confidence': 0.84,
            'reason': '学生知道从递推式入手，但不会主动提出构造新数列这个中间目标。',
            'evidence': '我知道要从递推式入手，但不知道为什么要构造 a_n+1/2。',
            'coach_mode': 'socratic_light',
            'coach_trap': '学生知道局部知识，但不会先确定中间目标。',
            'coach_prompt': '先给一个很小的起点，只透露下一步，不要直接给完整答案。',
            'source': 'manual_demo',
        },
        'coach_response': {
            'content': '我们先别急着算结果，先想这道题要把递推式改造成什么结构。你试着说说，新数列最想变成哪种前后项关系？',
            'reply_quality': 'empty',
            'strategy': {
                'mode': 'socratic_light',
                'trap': '学生知道局部知识，但不会先确定中间目标。',
                'prompt': '先让学生明确中间目标，再推进具体变形。',
            },
            'turn_index': 1,
            'done': False,
            'stop_reason': 'continue',
            'reply_analysis': {
                'quality': 'empty',
                'understands': False,
                'completed': False,
                'reason': '学生还说不出要把递推式改造成固定比的结构。',
            },
        },
        'review_result': 'wrong',
    },
    {
        'question_id': 'wq_demo_vec_001',
        'case': CASES[1],
        'binding': question_binding(
            'math.三角函数_平面向量与解三角形.平面向量.向量夹角公式',
            ['math.三角函数_平面向量与解三角形.平面向量.数量积几何意义'],
        ),
        'diagnosis': {
            'error_type': 'missing_strategy',
            'confidence': 0.78,
            'reason': '学生知道数量积，但不会把向量和关系先转成模平方，再落到夹角公式。',
            'evidence': '我能想到数量积，但不会把向量和的模平方展开到夹角公式。',
            'coach_mode': 'socratic_light',
            'coach_trap': '学生会局部公式，但不会先做结构转换。',
            'coach_prompt': '先提醒把向量和关系改写成模平方，再问下一步。',
            'source': 'manual_demo',
        },
        'coach_response': {
            'content': '这题不是直接套夹角公式，而是先把 a+b=-c 变成长度关系。你先试着说说，为什么这里先考虑 |a+b|^2 会更方便？',
            'reply_quality': 'weak',
            'strategy': {
                'mode': 'socratic_light',
                'trap': '学生会局部公式，但不会先做结构转换。',
                'prompt': '先点出结构转换，再只追问一个关键中间量。',
            },
            'turn_index': 1,
            'done': False,
            'stop_reason': 'continue',
            'reply_analysis': {
                'quality': 'weak',
                'understands': True,
                'completed': False,
                'reason': '学生已经能想到数量积，但还不会主动做模平方转换。',
            },
        },
        'review_result': 'partial',
    },
    {
        'question_id': 'wq_demo_circle_001',
        'case': CASES[2],
        'binding': question_binding(
            'math.解析几何.直线的方程.点到直线的距离公式',
            ['math.解析几何.圆的方程.圆的标准方程'],
        ),
        'diagnosis': {
            'error_type': 'concept_gap',
            'confidence': 0.81,
            'reason': '学生不会把张角存在性先转成圆心到直线距离范围，说明对几何判定条件理解不稳。',
            'evidence': '我知道要用点到直线距离，但不知道为什么先转成圆心到直线距离不超过 2。',
            'coach_mode': 'direct_explain',
            'coach_trap': '学生缺少必要几何判定概念，继续追问会放大挫败。',
            'coach_prompt': '先讲清“存在张角60°”为什么等价于圆心距离约束，再回到公式。',
            'source': 'manual_demo',
        },
        'coach_response': {
            'content': '这题的关键不是先代 t，而是先判断点 P 离圆心最远能到哪里。你先记住这里真正先要找的是“圆心到直线的距离范围”，再代入公式。',
            'reply_quality': 'good',
            'strategy': {
                'mode': 'direct_explain',
                'trap': '学生缺少必要几何判定概念，继续追问会放大挫败。',
                'prompt': '先补清概念判定，再回到距离公式。',
            },
            'turn_index': 1,
            'done': True,
            'stop_reason': 'student_understood',
            'reply_analysis': {
                'quality': 'good',
                'understands': True,
                'completed': True,
                'reason': '学生已经能接受先转成圆心到直线距离范围的思路。',
            },
        },
        'review_result': 'correct',
    },
]

SHOW_SCRIPT = []
for item in QUESTION_SCRIPTS:
    SHOW_SCRIPT.append(
        {
            'question_id': item['question_id'],
            'description': item['case']['description'],
            'diagnosis_error_type': item['diagnosis']['error_type'],
            'binding': item['binding'],
            'review_result': item['review_result'],
        }
    )

show_json(SHOW_SCRIPT)


[
  {
    "question_id": "wq_demo_seq_001",
    "description": "递推构造等比数列证明题",
    "diagnosis_error_type": "missing_strategy",
    "binding": {
      "primary_node_id": "math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化",
      "secondary_node_ids": [
        "math.数列与不等式.数列.基础概念.数列递推公式解读"
      ],
      "linked_node_ids": [
        "math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化",
        "math.数列与不等式.数列.基础概念.数列递推公式解读"
      ]
    },
    "review_result": "wrong"
  },
  {
    "question_id": "wq_demo_vec_001",
    "description": "单位向量和为零，借数量积与夹角公式求夹角",
    "diagnosis_error_type": "missing_strategy",
    "binding": {
      "primary_node_id": "math.三角函数_平面向量与解三角形.平面向量.向量夹角公式",
      "secondary_node_ids": [
        "math.三角函数_平面向量与解三角形.平面向量.数量积几何意义"
      ],
      "linked_node_ids": [
        "math.三角函数_平面向量与解三角形.平面向量.向量夹角公式",
        "math.三角函数_平面向量与解三角形.平面向量.数量积几何意义"
      ]
    },
    "review_result": "partial"
  },
  {
    "question_id": "wq_demo_circle_001",
    "description": "圆上张角存在性转化为圆心到直线距离范围",
    "diagnosis

## 3. 生成事件并写入 demo store

每道题我们都写三类事件：

- `binding_event`
- `diagnosis_event`
- `coach_event`
- `review_event`

最终这些事件会全部写到同一个 `student_memory_events_demo.jsonl` 文件里。

In [ ]:
reset_demo_store(DEMO_EVENTS_PATH)

WRITTEN_EVENTS = []

for index, item in enumerate(QUESTION_SCRIPTS):
    occurred_base = BASE_TIME + timedelta(minutes=index * 25)
    question = item['case']['question_payload']

    binding_event = binding_result_to_memory_event(
        question_id=item['question_id'],
        question_type=question.get('question_type'),
        primary_node_id=item['binding']['primary_node_id'],
        secondary_node_ids=item['binding']['secondary_node_ids'],
        candidate_node_ids=item['binding']['linked_node_ids'],
        binding_source='student_confirmed',
        occurred_at=occurred_base.isoformat(),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
        binding_confidence=0.9,
    )
    append_event(binding_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(binding_event)

    diagnosis_event = diagnosis_result_to_memory_event(
        item['diagnosis'],
        question_id=item['question_id'],
        question_type=question.get('question_type'),
        binding=item['binding'],
        occurred_at=(occurred_base + timedelta(minutes=2)).isoformat(),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(diagnosis_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(diagnosis_event)

    coach_event = coach_response_to_memory_event(
        item['coach_response'],
        question_id=item['question_id'],
        question_type=question.get('question_type'),
        binding=item['binding'],
        occurred_at=(occurred_base + timedelta(minutes=5)).isoformat(),
        error_type=item['diagnosis']['error_type'],
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(coach_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(coach_event)

    review_event = review_state_update_to_memory_event(
        {
            'action': 'review_result',
            'target_type': 'wrong_question',
            'target_id': item['question_id'],
            'result': item['review_result'],
            'student_id': STUDENT_ID,
            'session_id': SESSION_ID,
            'executed_at': (occurred_base + timedelta(days=2)).isoformat(),
            'updated_payload': {
                'example_question_states': [
                    {
                        'question_id': item['question_id'],
                        'question_type': question.get('question_type'),
                        'linked_node_ids': item['binding']['linked_node_ids'],
                        'primary_node_ids': [item['binding']['primary_node_id']],
                        'secondary_node_ids': item['binding']['secondary_node_ids'],
                        'last_result': item['review_result'],
                        'review_count': 1,
                    }
                ],
                'knowledge_point_states': [
                    {
                        'node_id': item['binding']['primary_node_id'],
                        'linked_question_ids': [item['question_id']],
                        'mastery': 0.25 if item['review_result'] == 'wrong' else 0.55 if item['review_result'] == 'partial' else 0.82,
                        'stability': 0.4 if item['review_result'] == 'wrong' else 1.2 if item['review_result'] == 'partial' else 2.8,
                    }
                ],
            },
        }
    )
    append_event(review_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(review_event)

print('demo store path:', DEMO_EVENTS_PATH)
print('written events:', len(WRITTEN_EVENTS))


In [ ]:
show_json(summarize_store(DEMO_EVENTS_PATH).as_dict())


## 4. 看看事件库里到底存了什么

先看全部事件，再按事件类型拆开看。

In [ ]:
ALL_EVENTS = load_events(path=DEMO_EVENTS_PATH, student_id=STUDENT_ID)
show_json(ALL_EVENTS)


In [ ]:
show_json(
    {
        'diagnosis_events': load_events(path=DEMO_EVENTS_PATH, student_id=STUDENT_ID, event_type='diagnosis'),
        'coach_events': load_events(path=DEMO_EVENTS_PATH, student_id=STUDENT_ID, event_type='coach'),
        'review_events': load_events(path=DEMO_EVENTS_PATH, student_id=STUDENT_ID, event_type='review'),
    }
)


## 5. 从事件库重建一个统一的 student_memory_profile

这里不再手工传 diagnosis / coach / review 列表，
而是直接从 `jsonl` 事件库里按学生读取并重建。

In [ ]:
PROFILE = build_profile_from_store(
    student_id=STUDENT_ID,
    path=DEMO_EVENTS_PATH,
)

show_json(PROFILE)


## 6. 重点看 profile 的几个核心部分

这里最值得关注的是：

- `error_type_counts`
- `node_memories`
- `question_memories`
- `personalization_summary`
- `agent_memory_text`


In [ ]:
show_json(
    {
        'error_type_counts': PROFILE.get('error_type_counts'),
        'node_memories': PROFILE.get('node_memories'),
        'question_memories': PROFILE.get('question_memories'),
        'personalization_summary': PROFILE.get('personalization_summary'),
        'agent_memory_text': PROFILE.get('agent_memory_text'),
    }
)


## 7. 你现在应该怎样理解这条链

这份 notebook 演示的是：

1. 一道题不是直接变成 profile
2. 而是先变成多条结构化事件
3. 这些事件统一写进 store
4. 最后再由 `student_memory_manager` 聚合成一个总 profile

也就是说：

- store 保存的是历史过程
- profile 保存的是当前总结

后面如果你接 `Cognee`，更像是把这份 store 换成更高级的底层记忆仓库，而不是跳过这一层。